# Sarcasm Detection - a guided walkthrough (runs on your laptop)

This notebook runs the whole project **on your own computer**, step by step, using
a **small sample** of comments so it finishes quickly. You can read what each step
does and see the results appear right here.

Run the cells from top to bottom (click a cell, press **Shift+Enter**).

> For the real, full-quality run, use `run_everything.py` instead (see the README).
> This notebook is for learning and quick checks.

## Step 0 - get ready

This cell moves into the project folder and chooses a **small** sample so the
transformers train in a minute or two instead of hours.

In [ ]:
import os

# When you open this notebook, the code starts in the 'notebooks' folder.
# Move up one level into the main 'project' folder so we can use the project's code.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working in:', os.getcwd())

# Choose a SMALL sample so this runs fast. (The real run uses much more.)
import settings
settings.SAMPLE_SIZE = 3000   # use only 3000 comments
settings.EPOCHS = 1           # train each transformer just once
print('Using', settings.SAMPLE_SIZE, 'comments and', settings.EPOCHS, 'epoch(s).')

## Step 1 - look at the data

Let's load the Reddit comments and see what they look like. Each row has a
`comment`, the `parent_comment` it replied to, and a `label` (1 = sarcastic).

In [ ]:
from engine.data import load_comments

comments = load_comments()
print('Loaded', len(comments), 'comments.')
print('Fraction sarcastic:', round(comments['label'].mean(), 2), '(about half - good)')

# Show the first few rows as a table.
comments.head()

In [ ]:
# Look at one real example of each kind.
sarcastic = comments[comments['label'] == 1].iloc[0]
normal    = comments[comments['label'] == 0].iloc[0]

print('A SARCASTIC comment:')
print('   they were replying to:', sarcastic['parent_comment'][:120])
print('   the comment          :', sarcastic['comment'][:120])
print()
print('A NOT-sarcastic comment:')
print('   they were replying to:', normal['parent_comment'][:120])
print('   the comment          :', normal['comment'][:120])

## Step 2 - the simple model (TF-IDF)

First the simple baseline: it turns words into numbers and learns to separate
sarcastic from non-sarcastic. We run it twice - with only the comment, and with the
context (the parent comment) added - to see if context helps.

In [ ]:
from engine.baseline import train_and_score_baseline

train_and_score_baseline(use_context=False)   # comment only
train_and_score_baseline(use_context=True)    # comment + context

## Step 3 - BERT

Now a real language model. BERT actually understands sentences, so it should do
better than the simple model. This takes a minute or two (it learns on the GPU if
you have one). Again we try it without and with context.

In [ ]:
from engine.transformer import train_and_score

train_and_score(model_name='bert-base-uncased', use_context=False)   # comment only
train_and_score(model_name='bert-base-uncased', use_context=True)    # comment + context

## Step 4 - RoBERTa

RoBERTa is a better-trained version of BERT. (The first time, it downloads the
model, which can take a moment.)

In [ ]:
train_and_score(model_name='roberta-base', use_context=False)
train_and_score(model_name='roberta-base', use_context=True)

## Step 5 - compare everything

This prints all the scores side by side, and checks whether adding context made a
**real** difference or could just be luck.

Note: with only 3000 comments the scores are low and wobbly - that is expected. The
point here is to see the whole thing work. Use a bigger `SAMPLE_SIZE` for real
results.

In [ ]:
from engine.scoring import show_table

show_table()

## Done!

To get **good** numbers, run the real version instead of this small sample:

- open a terminal in the `project` folder, then run `python run_everything.py`, or
- raise `SAMPLE_SIZE` in `settings.py` (e.g. 20000, or `None` for all the data) and
  re-run this notebook (use the menu: **Kernel -> Restart** first).

The transformer steps are slow without a GPU - for a big run, use Kaggle's free GPU
(see `../docs/kaggle.md`).